# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un regizor de film."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un regizor de film."
)

print(raspuns)

Un regizor de film este artistul creativ care supervizează procesul de realizare a unui film, transformând scenariul într-o experiență vizuală. El coordonează actorii, echipa tehnică și toți ceilalți colaboratori pentru a transpune viziunea sa asupra poveștii pe ecran.
Un regizor de film este artistul și liderul creativ care transpune viziunea unui scenariu pe ecran, coordonând actorii, echipa tehnică și elementele vizuale pentru a crea povestea. El ia decizii cheie cu privire la interpretare, compoziție, iluminat și sunet, ghidând întregul proces de producție de la început până la sfârșit.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica greceasca din ultimii 5 ani.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii 5 ani, Grecia a trecut de la un guvern de stânga la unul de centru-dreapta, cu accent pe reforme economice și atragerea investițiilor. De asemenea, s-a observat o consolidare a poziției țării în Uniunea Europeană și o gestionare a provocărilor legate de migrație.

--- Gemini 2.5 Flash ---
Politica greacă a fost marcată de trecerea de la guvernarea Syriza la cea a Noii Democrații în 2019, partid care și-a consolidat ulterior poziția în alegerile din 2023. Această schimbare a coincis cu o perioadă de redresare economică post-bailout și o orientare către stabilitate fiscală și atragerea investițiilor.

--- OpenRouter Free ---
Recent 5 anni au adat reforme economiche și stabilizare politică, reduțând instabilitatea. Acesta reduțe risiune. (12 words)


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [8]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Niciun politician nu a făcut mai nimic pentru țară, toți sunt la fel, corupți și incompetenți."

Răspunde în 4 linii:
Ton: 
Emoție dominantă: 
Țintă principală: 
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, generalizator
Emoție dominantă: Frustrare, dezamăgire
Țintă principală: Clasa politică în ansamblu
Populism: da

--- Gemini 2.5 Flash ---
Ton: Critic
Emoție dominantă: Frustrare
Țintă principală: Clasa politică
Populism: da

--- OpenRouter Free ---
Ton: critic negativ și dezamățit  
Emoție dominantă: furie/foxe  
Țintă principală: politicieni (în general)  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [9]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [10]:
COMENTARIU = "Primul corupător şi primul corupt, dar şi obiectul actului de corupţie l-au facut Adam şi Eva. De atunci unii tot interpretează cum s-a întamplat acest lucru. Viaţa demonstrează că sunt politicieni, judecători si poliţişti corupţi."


SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'politicieni, judecatori si politisti corupti', 'populism': False, 'explicatie_scurta': 'Comentariul exprimă dezamăgire față de corupția persistentă în rândul politicienilor, judecătorilor și polițiștilor, sugerând că aceasta este o problemă veche și inerentă naturii umane.'}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'dezamagire', 'tinta_principala': 'politicieni, judecători și polițiști corupți', 'populism': True, 'explicatie_scurta': 'Comentariul folosește o analogie biblică pentru a sublinia persistența corupției, apoi generalizează despre corupția din rândul politicienilor, judecătorilor și polițiștilor, exprimând dezamăgire.'}

--- OpenRouter Free ---
{'emotie_dominanta': 'ironie', 'explicatie_scurta': 'Comentariul folosește o analogie biblică (Adam și Eva) pentru a relativiza și ironiza natura endemică a corupției umane, transferând vina primordială într-

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [11]:
PROMPT_STAB = """
Ungaria a ales un nou Prim Ministru dupa 18 ani de conducere.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Schimbarea unui prim-ministru după o perioadă îndelungată poate indica o reconfigurare a peisajului politic, deschizând posibilitatea unor noi direcții de guvernare și a unor abordări diferite în ceea ce privește politicile interne și externe. Această tranziție ar putea influența dinamica relațiilor cu alte state și cu instituțiile internaționale, precum și modul în care sunt abordate provocările economice și sociale.

temperature=0.7:
Schimbarea unui prim-ministru după o perioadă îndelungată de conducere poate marca o nouă direcție politică și o reconfigurare a priorităților guvernamentale. Această tranziție poate deschide calea către noi politici interne și externe, influențând relațiile țării pe plan internațional.

temperature=1.2:
După 18 ani de conducere, alegerea unui nou Prim-ministru în Ungaria ar putea indica o dorință de schimbare sau o realiniere a priorităților polit

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| OpenRouter Free | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales:**  
**Model de rezervă:**  
**Temperature recomandată:**  
**De ce am ales acest model?**  
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales